# Gateway

The only code that builds a model client. Everything else receives `get_chat_model` as an argument
and never learns what a gateway is.

Two paths off the same workspace host: the AI Gateway serves the `finhive_router` and
`finhive_embeddings` services, and Databricks' own pay-per-token models answer on
`/serving-endpoints`, which is where the guardrails run because they are the most frequent and most
mechanical calls in the graph (`agent-design.md` §11.3). The host and the token both come from the
notebook context, so no workspace URL is written down and no PAT is stored.

## Install the dependencies first

This notebook imports `langchain_openai` at the top, so in a clean environment even `%run`-ing it
fails with `ModuleNotFoundError`. Once per session:

    %pip install -r ../requirements.txt
    dbutils.library.restartPython()

`restartPython` wipes the namespace, which is why this notebook installs nothing itself — doing so
would reset every caller that `%run` it. On serverless, pointing the notebook Environment panel at
`notebooks/agents/requirements.txt` does the same thing without the restart.

## Running the check

`check()` is defined but not called — it costs nine model calls, so `%run`-ing this notebook has to
stay free. Run `check()` in a cell to exercise it.

It separates rows that **must** pass from rows that are **diagnostic**. The two direct probes
against `MODELS_ROUTED` use Model Serving endpoint names that have not been verified against this
workspace (§22 item 10), so a 404 there reports itself and does not fail the run.

In [ ]:
%run ../config

In [ ]:
%run ./parsing

In [ ]:
try:
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
except ModuleNotFoundError as exc:  # the first-run failure; say how to fix it, do not hide it
    raise ModuleNotFoundError(
        f"{exc.name} is not installed. Run this once per session, then re-run this notebook: "
        "`%pip install -r ../requirements.txt` followed by `dbutils.library.restartPython()`"
    ) from exc

# role -> (path, model). The only place a role becomes a model name.
ROLE_MODELS = {
    "router": (AI_GATEWAY_PATH, MODEL_ROUTER),
    "worker": (AI_GATEWAY_PATH, MODEL_ROUTER),
    "synthesizer": (AI_GATEWAY_PATH, MODEL_ROUTER),
    "guard_in": (SERVING_PATH, MODEL_GUARD_IN),
    "guard_out": (SERVING_PATH, MODEL_GUARD_OUT),
    "embedding": (AI_GATEWAY_PATH, MODEL_EMBEDDINGS),
}

# §21: reasoning models spent the whole output budget thinking and returned empty content in ~40%
# of calls. Floor every chat call.
MIN_OUTPUT_TOKENS = 300

_context = None


def _ctx():
    """Host and token both come from the notebook context, cached. No PAT, no hardcoded workspace."""
    global _context
    if _context is None:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        _context = (ctx.apiUrl().get().rstrip("/"), ctx.apiToken().get())
    return _context


def _chat(path, model, temperature, max_tokens):
    host, token = _ctx()
    return ChatOpenAI(
        model=model,
        base_url=f"{host}{path}",
        api_key=token,
        temperature=temperature,
        timeout=MODEL_TIMEOUT_SECONDS,
        # langchain-openai rewrites max_tokens to max_completion_tokens, which the gateway rejects
        # with 400 unknown field (§21). Pass the cap through untouched.
        extra_body={"max_tokens": max(max_tokens, MIN_OUTPUT_TOKENS)},
    )


def get_chat_model(role, temperature=None, max_tokens=None):
    if role not in ROLE_MODELS or role == "embedding":
        raise ValueError(f"unknown chat role {role!r}; known: "
                         f"{sorted(r for r in ROLE_MODELS if r != 'embedding')}")
    path, model = ROLE_MODELS[role]
    settings = ROLE_SETTINGS[role]
    return _chat(path, model,
                 settings["temperature"] if temperature is None else temperature,
                 max_tokens or settings["max_tokens"])


def get_embedding_model():
    path, model = ROLE_MODELS["embedding"]
    host, token = _ctx()
    # check_embedding_ctx_length would try to tokenize for an OpenAI model name that is not one
    return OpenAIEmbeddings(model=model, base_url=f"{host}{path}", api_key=token,
                            check_embedding_ctx_length=False)

In [ ]:
from pydantic import BaseModel, Field


class _Probe(BaseModel):
    """Flat, every field required - the shape §4.2 requires of every schema."""
    answer: str = Field(description="the capital city")
    confident: bool = Field(description="whether the answer is certain")


def _structured(label, chat, ask, fallback):
    """One structured round trip -> (label, ok, detail). Never raises."""
    try:
        got = ask_structured(chat, *ask, _Probe, fallback)
        return (label, bool(got.answer.strip()),
                f"{got.answer!r} confident={got.confident}" if got.answer.strip()
                else "fell back - response_format not honoured, use function calling")
    except Exception as exc:
        return label, False, f"{type(exc).__name__}: {exc}"


def check():
    """Seven live calls, plus two more if MODELS_ROUTED is filled in. Raises if a required row failed.

    Every chat role answers, embeddings return a vector, and structured output is probed through the
    router service. That last one is necessary but not sufficient: the router picks a model per
    request, so one round trip through it exercises one of the two with 70/30 odds and would pass
    while the graph fails intermittently in production (agent-design.md §22 item 11). Probing each
    routed model *directly* is what settles it, and that needs their Model Serving endpoint names -
    which are not the Unity Catalog model names the router is configured with, so MODELS_ROUTED
    ships empty and those two probes stay off until someone fills it in (§22 item 10). They report
    as diagnostic either way: an endpoint name is a name to fix, not a system that is down.
    """
    rows = []
    fallback = _Probe(answer="", confident=False)
    ask = ("Answer about world capitals.", "What is the capital of France?")

    for role in (r for r in ROLE_MODELS if r != "embedding"):
        try:
            text = message_text(get_chat_model(role).invoke(
                [{"role": "user", "content": "Reply with the single word: ready"}]))
            rows.append((role, bool(text.strip()), f"{ROLE_MODELS[role][1]} -> {text.strip()[:60]!r}"))
        except Exception as exc:
            rows.append((role, False, f"{type(exc).__name__}: {exc}"))

    try:
        vector = get_embedding_model().embed_query("What is Databricks?")
        rows.append(("embedding", len(vector) > 0, f"{MODEL_EMBEDDINGS} -> dim {len(vector)}"))
    except Exception as exc:
        rows.append(("embedding", False, f"{type(exc).__name__}: {exc}"))

    unsupported = check_schema_supported(_Probe)
    rows.append(("probe_schema", not unsupported,
                 f"unsupported={unsupported}" if unsupported else "flat, all required"))

    rows.append(_structured("structured/router", get_chat_model("router"), ask, fallback))
    diagnostic = [_structured(f"structured/{model}", _chat(SERVING_PATH, model, 0.0, 600),
                              ask, fallback)
                  for model in MODELS_ROUTED]

    for label, ok, detail in rows:
        print(f"{'PASS' if ok else 'FAIL'}  {label:<44}  {detail}")

    print(chr(10) + "diagnostic, never a failure - an endpoint name is a name to fix:")
    if diagnostic:
        for label, ok, detail in diagnostic:
            print(f"{'pass' if ok else '....'}  {label:<44}  {detail}")
    else:
        print("....  MODELS_ROUTED is empty, so the 70/30 probe is off. Until it is filled in, a")
        print("      green structured/router row proves one of the two routed models, not both.")

    failed = [label for label, ok, _ in rows if not ok]
    if failed:
        raise RuntimeError(f"llm/gateway check failed: {failed}")
    return {"ok": True, "rows": rows, "diagnostic": diagnostic}